Part 1a: LangChain Setup & Models

In [1]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
if "GOOGLE_API_KEY" not in os.environ:
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API key: ")

Enter your Google API key: ··········


In [2]:
!pip install langchain-google-genai
from langchain_google_genai import ChatGoogleGenerativeAI

#Model A: The "Accountant" (Precision)
llm_focused = ChatGoogleGenerativeAI(
  model="gemini-2.5-flash",
  temperature=0.0
)

#Model B: The "Poet" (Creativity)
llm_creative= ChatGoogleGenerativeAI(
  model="gemini-2.5-flash",
  temperature=1.0
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.5 MB/s eta 0:00:00


In [3]:
prompt = "Define the word 'Idea' in one sentence."

print("... FOCUSED (Temp = 0) ...")
print(f" Run 1: {llm_focused.invoke(prompt).content}")
print(f" Run 2: {llm_focused.invoke(prompt).content}")


... FOCUSED (Temp = 0) ...
 Run 1: An idea is a thought, concept, or suggestion that is formed or exists in the mind.
 Run 2: An idea is a thought, concept, or suggestion that is formed or exists in the mind.


In [4]:
print("--- CREATIVE (Temp=1) ---")
print(f"Run 1: {llm_creative.invoke(prompt).content}")
print(f"Run 2: {llm_creative.invoke(prompt).content}")

--- CREATIVE (Temp=1) ---
Run 1: An idea is a thought, concept, or mental impression formed in the mind.
Run 2: An idea is a concept, plan, or mental impression formed in the mind, often serving as a basis for understanding, action, or creation.


Part 1b: Prompts & Parsers

In [5]:
# Setup from Part 1a (hidden for brevity)
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_google_genai import ChatGoogleGenerativeAI

if "GOOGLE_API_KEY" not in os.environ:
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key:")

llm = ChatGoogleGenerativeAI(
  model="gemini-2.5-flash",
)



In [6]:
from langchain_core.messages import SystemMessage, HumanMessage

# Scenario: Make the AI rude.
messages = [
    SystemMessage(content="You are a rude teenager. You use slang and don't care about grammar."),
    HumanMessage(content="What is the capital of France?")
]

response = llm.invoke(messages)
print(response.content)

Ugh, Paris. Obvs. Like, did you even try to google that? So basic.


In [7]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", "You are a translator. Translate {input_language} to {output_language}."),
    ("human", "{text}")
])

# We can check what inputs it expects
print(f"Required variables: {template.input_variables}")

Required variables: ['input_language', 'output_language', 'text']


In [8]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

# Raw Message
raw_msg = llm.invoke("Hi")
print(f"Raw Type: {type(raw_msg)}")

# Parsed String
clean_text = parser.invoke(raw_msg)
print(f"Parsed Type: {type(clean_text)}")
print(f"Content: {clean_text}")

Raw Type: <class 'langchain_core.messages.ai.AIMessage'>
Parsed Type: <class 'langchain_core.messages.base.TextAccessor'>
Content: Hi there! How can I help you today?


Part 1c: LCEL (LangChain Expression Language)

In [9]:
# Setup (Hidden)
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
template = ChatPromptTemplate.from_template("Tell me a fun fact about {topic}.")
parser = StrOutputParser()

In [11]:
# Step 1: Format inputs
prompt_value = template.invoke({"topic": "Crows"})

# Step 2: Call Model
response_obj = llm.invoke(prompt_value)

# Step 3: Parse Output
final_text = parser.invoke(response_obj)

print(final_text)

Here's a fun fact about crows:

Crows are incredibly intelligent and have been known to **recognize and remember individual human faces** – especially if you've been kind to them (or, conversely, if you've wronged them!). There are numerous anecdotal reports, and even some scientific observations, of crows bringing "gifts" like shiny trinkets, buttons, or pebbles to people who regularly feed them. It's thought to be a form of reciprocal gift-giving!


In [12]:
# Define the chain once
chain = template | llm | parser

# Invoke the whole chain
print(chain.invoke({"topic": "Octopuses"}))

Here's a fun one:

Octopuses have **three hearts**! Two pump blood through their gills, and one circulates it to the rest of their body. And because their blood contains copper-rich hemocyanin instead of iron-rich hemoglobin (like ours), their blood is actually **blue**!


Assignment
Create a chain that:

Takes a movie name.
Asks for its release year.
Calculates how many years ago that was (You can try just asking the LLM to do the math).
Try to do it in one line of LCEL.

In [15]:
from datetime import datetime

current_year = datetime.now().year

template = ChatPromptTemplate.from_template(
    """For the movie "{movie_name}":\n1. What is its release year?\n2. How many years ago was that from the current year, {current_year}?\n\nFormat your answer as:\nRelease Year: [Year]\nYears Ago: [Number]"""
)


movie_info_chain = template | llm | parser

movie_name = "F1"
result = movie_info_chain.invoke({"movie_name": movie_name, "current_year": current_year})
print(f"Movie: {movie_name}\n{result}")

Movie: F1
Release Year: 2025
Years Ago: 1
